# **PROJECT 7:** AI-Powered Plant Disease Detection with Generative AI Explanations

### **Step-1: Upload the saved model zip file (downloaded from Teachable Machine) in Google Colab**

In [7]:
#@title Upload the saved model zip file and Run this cell to Unzip it

import os
from IPython.display import clear_output

# Get all .zip files in the current folder
zip_files = [f for f in os.listdir('.') if f.endswith('.zip')]

if not zip_files:
    print("Please upload the saved model zip file (downloaded from Teachable Machine) in Google Colab.")
else:
    !unzip "$(ls | grep '.zip')"
    clear_output()
    print("Model Unzipped successfully!")

Model Unzipped successfully!


### **Step-2: Load Model and Class Labels**

In [8]:
#@title Run this cell to Import required libraries

import tensorflow as tf
from PIL import Image, ImageOps
import numpy as np
# Disable scientific notation for clarity
np.set_printoptions(suppress=True)

print("Libraries imported successfully!")

Libraries imported successfully!


In [9]:
#@title Run this cell to Load the SavedModel

# ----------------------------
# Load SavedModel
# ----------------------------
saved_model = tf.saved_model.load("model.savedmodel")

# Get the serving function
infer = saved_model.signatures["serving_default"]

print("Model loaded successfully!")

Model loaded successfully!


In [10]:
#@title Run this cell to Load Class labels

class_names = open("labels.txt", "r").readlines()
class_names = [name.split()[-1] for name in class_names]

print("Class labels loaded successfully!")
print(class_names)

Class labels loaded successfully!
['Apple_Scab', 'Apple_Healthy', 'Potato_Early_Blight', 'Potato_Healthy', 'Tomato_Leaf_Mold', 'Tomato_Healthy']


### **Step-3: Upload a Test image and Get Prediction**

In [14]:
#@title Upload a Plant Leaf Test image, provide its file name, then Run this cell

# "Upload a test image in Colab and provide its file name here (eg. Apple_scab_test_1.JPG)"
img_file = 'Plant_leave_diseases_dataset\\Testing\\Apple_healthy_test_2.JPG'  #@param {type:"string"}

# Prepare the input image
data = np.ndarray(shape=(1, 224, 224, 3), dtype=np.float32)

image = Image.open(img_file).convert("RGB")
size = (224, 224)
image = ImageOps.fit(image, size, Image.Resampling.LANCZOS)

image_array = np.asarray(image)
normalized_image_array = (image_array.astype(np.float32) / 127.5) - 1
data[0] = normalized_image_array

# Show image
image_array

array([[[165, 153, 189],
        [165, 153, 189],
        [165, 153, 189],
        ...,
        [177, 171, 207],
        [177, 171, 207],
        [177, 171, 207]],

       [[165, 153, 189],
        [165, 153, 189],
        [165, 153, 189],
        ...,
        [176, 170, 206],
        [176, 170, 206],
        [176, 170, 206]],

       [[165, 153, 189],
        [165, 153, 189],
        [165, 153, 189],
        ...,
        [174, 168, 204],
        [174, 168, 204],
        [174, 168, 204]],

       ...,

       [[144, 128, 165],
        [137, 121, 158],
        [135, 119, 156],
        ...,
        [138, 128, 163],
        [143, 133, 168],
        [147, 137, 172]],

       [[137, 121, 158],
        [131, 115, 152],
        [130, 114, 151],
        ...,
        [141, 131, 166],
        [145, 135, 170],
        [142, 132, 167]],

       [[138, 122, 159],
        [133, 117, 154],
        [136, 120, 157],
        ...,
        [149, 139, 174],
        [148, 138, 173],
        [140, 130, 165]]

In [13]:
import glob
images = glob.glob("Plant_leave_diseases_dataset/**/*.[jJ][pP]*[gG]", recursive=True)
print(len(images))
print(images[:5])

204
['Plant_leave_diseases_dataset\\Testing\\Apple_healthy_test_1.JPG', 'Plant_leave_diseases_dataset\\Testing\\Apple_healthy_test_2.JPG', 'Plant_leave_diseases_dataset\\Testing\\Apple_healthy_test_3.JPG', 'Plant_leave_diseases_dataset\\Testing\\Apple_healthy_test_4.JPG', 'Plant_leave_diseases_dataset\\Testing\\Apple_scab_test_1.JPG']


In [15]:
#@title Run this cell to Get Prediction for the test image

# ----------------------------
# Run Prediction
# ----------------------------
# Convert numpy → tensor
input_tensor = tf.convert_to_tensor(data)

# Call the SavedModel signature
output = infer(input_tensor)

# SavedModel may return a dict – get the first output tensor
prediction = list(output.values())[0].numpy()

# Decode prediction
index = np.argmax(prediction)
class_name = class_names[index].strip()
confidence_score = prediction[0][index]

# Print results
print("Class:", class_name)
print("Confidence Score:", confidence_score)


Class: Apple_Healthy
Confidence Score: 0.9998135


### **Step-4: Provide the Google Gemini API Key**

* Use an existing Google Gemini API key, or Create a new one by:
    * Visiting https://ai.google.dev/gemini-api/docs, and
    * Clicking on *Get a Geminin API key*
    * Note that Google Gemini's Free plan includes limited access to certain models



In [16]:
#@title Enter your Gemini API Key
import os

GEMINI_API_KEY = input("Enter your Gemini API Key: ")
os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY



### **Step-5: Define prompt to feed to an LLM to get farmer-friendly explanations, treatment recommendations, and optional multilingual support for regional accessibility.**

Adjust the prompt as required.

In [17]:
# ----------------------------
# Define Prompt
# ----------------------------

language = "English"

# Prompt For Response Generation
# Use a fallback so this cell still works even if the prediction cell hasn't run yet
predicted_disease = globals().get("class_name", "Unknown disease")

prompt = f"""
You are an expert Agricultural plant pathologist and crop-health advisor.
Explain the condition {predicted_disease} and provide practical, safe, and easy-to-apply remedies.

Include:

- A brief explanation of the disease
- Key symptoms
- Immediate actions a farmer/gardener should take
- Natural or low-cost remedies
- When to consider chemical treatments
- Preventive measures for future crops

Symptom: {predicted_disease}

Answer only in {language}.

If the plant is unhealthy, then start the response in this format:
`**Plant:** <plant-name>
 **Status:** Unhealthy
 **Disease:** <disease-name>`

If the plant is healthy, then respond in this format:
`**Plant:** <plant-name>
 **Status:** Healthy`
"""

print("\nPROMPT\n" + "="*100)
print(prompt)


PROMPT

You are an expert Agricultural plant pathologist and crop-health advisor.
Explain the condition Apple_Healthy and provide practical, safe, and easy-to-apply remedies.

Include:

- A brief explanation of the disease
- Key symptoms
- Immediate actions a farmer/gardener should take
- Natural or low-cost remedies
- When to consider chemical treatments
- Preventive measures for future crops

Symptom: Apple_Healthy

Answer only in English.

If the plant is unhealthy, then start the response in this format:
`**Plant:** <plant-name>
 **Status:** Unhealthy
 **Disease:** <disease-name>`

If the plant is healthy, then respond in this format:
`**Plant:** <plant-name>
 **Status:** Healthy`



### **Step-6: Make an LLM API call using the above prompt**

In [18]:
#@title Run this cell to Make an LLM API Call

# ----------------------------
# Make an LLM Call
# ----------------------------

# Initialize Client
!pip install -q google-genai
import os
# Install the SDK in the current notebook environment
!pip install -q google-genai

# Import the SDK
# Install the SDK in the current notebook environment
%pip install -q google-genai

from google import genai

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
gemini_client = genai.Client()


# Make API call and get response
gemini_response = gemini_client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
)

# Show response in Markdown
from IPython.display import Markdown

Markdown(gemini_response.text)


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


**Plant:** Apple
**Status:** Healthy

As an expert Agricultural plant pathologist and crop-health advisor, I'm delighted to discuss the condition of an "Apple_Healthy" plant. This isn't a disease; rather, it signifies a thriving apple tree that is robust, productive, and free from significant pest infestations or pathogen infections.

### Explanation of the Condition (Health)

A healthy apple tree is in a state of optimal physiological function. Its immune system is strong, its nutrient uptake is efficient, and it is well-adapted to its growing environment. This allows it to produce high-quality fruit and maintain vigorous growth without external stress factors compromising its health. The goal of all horticultural practices is to maintain this healthy state.

### Key Symptoms (Indicators of Health)

*   **Vigorous Growth:** Strong shoot elongation, healthy bud development, and an overall robust appearance.
*   **Lush, Green Foliage:** Leaves are uniformly green (specific shade varies by cultivar), free from spots, discoloration, holes, wilting, or premature dropping. They are firm and turgid.
*   **Strong Branch Structure:** Well-formed scaffold branches, good branch angles, and no signs of dieback or cankers.
*   **Abundant Bloom and Fruit Set:** Healthy trees produce a good number of flowers leading to a satisfactory crop of developing fruit, appropriate for the cultivar and age of the tree.
*   **Clean, Unblemished Fruit:** Apples are well-formed, uniform in size and color for the variety, and free from scabs, rot, insect damage (e.g., codling moth entry holes, aphid clusters), or physiological disorders (e.g., bitter pit, cork spot).
*   **Healthy Root System:** While not directly visible, a healthy tree implies a well-developed, disease-free root system anchoring the tree and efficiently absorbing water and nutrients.
*   **Absence of Pests and Pathogens:** No visible signs of insect pests (aphids, mites, borers) or fungal/bacterial diseases (powdery mildew, apple scab, fire blight).

### Immediate Actions a Farmer/Gardener Should Take (To Maintain Health)

When your apple tree is healthy, the immediate action is to continue with good cultural practices and vigilant monitoring to *maintain* this ideal state.

1.  **Continue Routine Monitoring:** Regularly inspect your trees (at least weekly during the growing season) for any subtle changes that might indicate the onset of a problem. Pay close attention to new growth, the underside of leaves, and developing fruit.
2.  **Ensure Proper Watering:** Provide consistent and adequate water, especially during dry spells and fruit development. Deep watering encourages deep root growth. Avoid overwatering.
3.  **Maintain Good Sanitation:** Remove any fallen leaves or fruit from previous seasons, as these can harbor overwintering pests and disease spores. Keep the area around the tree free of weeds.
4.  **Observe for Early Warning Signs:** Even in a healthy tree, a few aphids or a small spot might appear. Catching these early allows for simple, non-chemical interventions.

### Natural or Low-Cost Remedies (Preventive Maintenance)

These are ongoing practices to support the tree's natural defenses and prevent problems from developing:

*   **Proper Pruning:** Prune annually during the dormant season to maintain an open canopy for good air circulation and light penetration, which reduces humidity and discourages fungal diseases. Remove dead, diseased, or crossing branches.
*   **Balanced Nutrition:** Conduct soil tests every 3-5 years to determine nutrient needs. Fertilize appropriately based on test results, typically with a balanced organic fertilizer or compost, in late winter/early spring. Avoid excessive nitrogen, which can lead to lush, susceptible growth.
*   **Mulching:** Apply a 2-4 inch layer of organic mulch (compost, wood chips) around the base of the tree, keeping it a few inches away from the trunk. This conserves soil moisture, regulates soil temperature, suppresses weeds, and adds organic matter.
*   **Encourage Beneficial Insects:** Plant diverse flowering plants nearby to attract natural predators of common apple pests (e.g., ladybugs, lacewings, parasitic wasps).
*   **Dormant Oil Application:** A horticultural oil spray applied during dormancy can suffocate overwintering insect eggs and scale insects. This is a low-impact, preventative measure.
*   **Physical Removal:** For minor pest outbreaks (e.g., a few aphids), hand-picking or a strong spray of water can dislodge them.

### When to Consider Chemical Treatments

For a healthy apple tree, chemical treatments should generally **not be considered** unless there is a clear and confirmed threat that cannot be managed through cultural practices or biological controls. The principle is **Integrated Pest Management (IPM)**:

*   **Identification is Key:** First, accurately identify the specific pest or disease. Misdiagnosis can lead to ineffective and potentially harmful treatments.
*   **Threshold Levels:** Only consider treatment if pest populations reach damaging levels or disease symptoms are progressing significantly. A few insects or minor leaf spots do not always warrant chemical intervention.
*   **Least Toxic Option First:** If intervention is necessary, always start with the least toxic and most targeted options (e.g., horticultural oils, insecticidal soaps, biological pesticides like Bt for specific caterpillars).
*   **Targeted Application:** Use chemicals only on affected areas, not as a broad-spectrum preventative on a healthy tree, which can harm beneficial insects and lead to resistance.
*   **Environmental Considerations:** Be aware of potential impacts on pollinators, water sources, and non-target organisms. Always follow label instructions precisely.
*   **Professional Consultation:** If you are unsure about a developing problem or treatment options, consult with a local extension office or certified arborist/plant pathologist.

### Preventive Measures for Future Crops

Maintaining a healthy apple tree is the best form of prevention for future crops. Continue the practices outlined above, and also consider:

*   **Disease-Resistant Cultivars:** When planting new trees, select cultivars known for resistance to common apple diseases in your region (e.g., scab-resistant varieties like 'Liberty' or 'Pristine').
*   **Proper Site Selection and Planting:** Ensure good soil drainage and adequate sunlight (at least 6-8 hours direct sun daily). Plant trees at the correct depth.
*   **Appropriate Spacing:** Allow enough space between trees for air circulation and light penetration, reducing disease pressure.
*   **Tool Sanitation:** Clean and sanitize pruning tools between trees (and even between major cuts on the same tree if dealing with a known disease) to prevent the spread of pathogens.
*   **Monitor Regional Forecasts:** Stay informed about local weather patterns that favor disease development (e.g., prolonged periods of leaf wetness for apple scab) and pest emergence.
*   **Crop Rotation (if applicable in orchard setting):** While not typically for perennial apple trees, managing cover crops and understory plants can contribute to overall orchard health.

### **Step-7: Create a Gradio Interface**

Use the Gemini present within Google Colab notebook ( <img src='https://drive.google.com/uc?id=19N2vjh7eihxQ3qj9uavYblyveidsLKsf' width=30> ), to generate code for creating a Gradio Interface with below features.



Inputs:
- `Gradio File` to let user upload an image file
- `Gradio Textbox` to let user enter his preferred Language

Outputs:
- `Gradio Image` to show the image uploaded by user
- `Gradio Markdown` to render the LLM response

Logic:
- The user input image should go to the plant disease detection model (trained on Teachable Machine)
- The predicted disease-label and the user preferred language should become part of the prompt
- Use an LLM to generate farmer-friendly explanations, treatment guidance, and preventive steps


In [ ]:
# ============================================================
# AI PLANT DISEASE DETECTION + GEMINI EXPLANATION + GRADIO
# ============================================================

import os
import sys
import subprocess

# ------------------------------------------------------------
# 1. Import required libraries
# ------------------------------------------------------------

try:
    import tensorflow as tf
except ModuleNotFoundError:
    print("TensorFlow not found. Installing it...")
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "tensorflow"]
        )
    except subprocess.CalledProcessError as e:
        raise RuntimeError(
            "TensorFlow installation failed. Please install it manually with: "
            "pip install tensorflow"
        ) from e

    import tensorflow as tf
import numpy as np
import gradio as gr

from PIL import Image, ImageOps
from google import genai


# ------------------------------------------------------------
# 2. Gemini API Key
# ------------------------------------------------------------
# Make sure GEMINI_API_KEY was created in an earlier cell.
#
# Example:
# GEMINI_API_KEY = "YOUR_API_KEY_HERE"
#
# For safety, don't upload your actual API key to GitHub.

if "GEMINI_API_KEY" not in globals():
    if "GEMINI_API_KEY" not in globals():
        GEMINI_API_KEY = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")

        if not GEMINI_API_KEY:
            GEMINI_API_KEY = input("Enter your Gemini API key and press Enter: ").strip()

        if not GEMINI_API_KEY:
            raise ValueError(
                "GEMINI_API_KEY is not defined. "
                "Please create your Gemini API key variable first."
            )

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

gemini_client = genai.Client(
    api_key=GEMINI_API_KEY
)


# ------------------------------------------------------------
# 3. CHECK MODEL
# ------------------------------------------------------------
# IMPORTANT:
# Your trained TensorFlow model must already be loaded.
#
# The variables expected here are:
#
#     infer
#     class_names
#
# If you already have a previous cell that creates these,
# run that cell before this code.
# ------------------------------------------------------------

if "infer" not in globals():
    if "infer" not in globals():
        model_candidates = [
            "keras_model.h5",
            "model.h5",
            "model.keras",
            "saved_model",
            os.path.join(os.getcwd(), "keras_model.h5"),
            os.path.join(os.getcwd(), "model.h5"),
            os.path.join(os.getcwd(), "model.keras"),
            os.path.join(os.getcwd(), "saved_model"),
        ]

        for candidate in model_candidates:
            if os.path.exists(candidate):
                try:
                    infer = tf.keras.models.load_model(candidate, compile=False)
                    print(f"Loaded TensorFlow model from: {candidate}")
                    break
                except Exception as exc:
                    print(f"Could not load model from {candidate}: {exc}")

        if "infer" not in globals():
            raise NameError(
                "The variable 'infer' was not found.\n\n"
                "Run the previous cell that loads your trained TensorFlow model before running this cell.\n"
                "Example: infer = tf.keras.models.load_model('keras_model.h5')"
            )

    if "class_names" not in globals():
        labels_candidates = [
            "labels.txt",
            os.path.join(os.getcwd(), "labels.txt"),
        ]

        for labels_path in labels_candidates:
            if os.path.exists(labels_path):
                with open(labels_path, "r", encoding="utf-8") as f:
                    class_names = [line.strip() for line in f if line.strip()]
                break

        if "class_names" not in globals():
            raise NameError(
                "The variable 'class_names' was not found.\n\n"
                "Run the previous cell that defines your disease class names before running this cell."
            )

if "class_names" not in globals():
    raise NameError(
        "The variable 'class_names' was not found.\n\n"
        "Run the previous cell that defines your disease "
        "class names before running this cell."
    )


# ------------------------------------------------------------
# 4. Prediction + Gemini explanation function
# ------------------------------------------------------------

def predict_disease_and_explain(image_file, language_input):

    # Check image
    if image_file is None:
        return None, "Please upload a plant leaf image."

    # Default language
    if not language_input:
        language_input = "English"


    # --------------------------------------------------------
    # Prepare image
    # --------------------------------------------------------

    data = np.ndarray(
        shape=(1, 224, 224, 3),
        dtype=np.float32
    )

    # Gradio may provide a filepath
    if isinstance(image_file, str):
        image_path = image_file
    else:
        image_path = image_file.name

    image = Image.open(image_path).convert("RGB")

    size = (224, 224)

    image_resized = ImageOps.fit(
        image,
        size,
        Image.Resampling.LANCZOS
    )

    image_array = np.asarray(image_resized)

    # Normalize exactly as expected by many Teachable Machine models
    normalized_image_array = (
        image_array.astype(np.float32) / 127.5
    ) - 1

    data[0] = normalized_image_array


    # --------------------------------------------------------
    # Run TensorFlow prediction
    # --------------------------------------------------------

    input_tensor = tf.convert_to_tensor(data)

    output = infer(input_tensor)

    # Handle dictionary output
    if isinstance(output, dict):
        prediction = list(output.values())[0].numpy()
    else:
        prediction = output.numpy()

    # Make sure prediction is 1D
    prediction = np.asarray(prediction).flatten()

    index = int(np.argmax(prediction))

    class_name = str(class_names[index]).strip()

    confidence_score = float(prediction[index])


    # --------------------------------------------------------
    # Determine healthy / unhealthy
    # --------------------------------------------------------

    if "Healthy" in class_name:

        plant_name = class_name.replace(
            "_Healthy", ""
        )

        status = "Healthy"

        prompt_for_llm = f"""
You are an expert agricultural plant pathologist
and crop-health advisor.

The AI image classifier identified the plant condition as:

{class_name}

Explain this result in simple language that a farmer
or gardener can understand.

Include:

- A brief explanation of the healthy condition
- What it means for the plant to be healthy
- General tips for maintaining plant health
- How to prevent common plant diseases

Plant: {plant_name}
Classification confidence: {confidence_score * 100:.2f}%

Answer ONLY in {language_input}.

Start your response with:

**Plant:** {plant_name}
**Status:** Healthy
**Confidence:** {confidence_score * 100:.2f}%
"""


    else:

        parts = class_name.split("_")

        if len(parts) > 1:
            plant_name = parts[0]
        else:
            plant_name = "Plant"

        disease_name = class_name

        status = "Unhealthy"

        prompt_for_llm = f"""
You are an expert agricultural plant pathologist
and crop-health advisor.

The AI image classifier identified the plant condition as:

{class_name}

Provide practical, safe, and easy-to-understand
information for a farmer or gardener.

Include:

- A brief explanation of the disease
- Key symptoms
- Immediate actions to take
- Natural or low-cost remedies
- When chemical treatment may be appropriate
- Preventive measures for future crops

Do not claim that the AI diagnosis is 100% certain.
Mention that the result should be confirmed by an
agricultural expert if the plant is valuable or the
symptoms are unclear.

Plant: {plant_name}
Disease: {disease_name}
Classification confidence: {confidence_score * 100:.2f}%

Answer ONLY in {language_input}.

Start your response with:

**Plant:** {plant_name}
**Status:** Unhealthy
**Disease:** {disease_name}
**Confidence:** {confidence_score * 100:.2f}%
"""


    # --------------------------------------------------------
    # Gemini API call
    # --------------------------------------------------------

    try:

        gemini_response = gemini_client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt_for_llm
        )

        explanation = gemini_response.text

    except Exception as e:

        explanation = (
            f"### Prediction\n\n"
            f"**Plant:** {plant_name}\n\n"
            f"**Status:** {status}\n\n"
            f"**Disease/Class:** {class_name}\n\n"
            f"**Confidence:** "
            f"{confidence_score * 100:.2f}%\n\n"
            f"### Gemini explanation unavailable\n\n"
            f"Error: `{str(e)}`"
        )


    # --------------------------------------------------------
    # Return results to Gradio
    # --------------------------------------------------------

    return image, explanation


# ------------------------------------------------------------
# 5. Create Gradio interface
# ------------------------------------------------------------

iface = gr.Interface(

    fn=predict_disease_and_explain,

    inputs=[

        gr.File(
            label="🌿 Upload a Plant Leaf Image",
            file_types=[".jpg", ".jpeg", ".png"]
        ),

        gr.Textbox(
            label="🌐 Enter Preferred Language",
            value="English",
            placeholder="Example: English, Tamil, Hindi..."
        )
    ],

    outputs=[

        gr.Image(
            label="📷 Uploaded Plant Image"
        ),

        gr.Markdown(
            label="🤖 AI-Powered Disease Explanation"
        )
    ],

    title="🌱 AI Plant Disease Detection",

    description=(
        "Upload a plant leaf image to detect possible plant "
        "diseases and receive an AI-generated explanation "
        "and practical guidance in your preferred language."
    ),

    flagging_mode="never"
)


# ------------------------------------------------------------
# 6. Launch application
# ------------------------------------------------------------

iface.launch(share=True, debug=True)

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://3bce2675795a2a04ac.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import tensorflow as tf
print(tf.__version__)

2.21.0


This code creates a simple Gradio interface:

*   `upload_image` is a function that takes the uploaded file object and returns its name.
*   `gr.File(label="Upload an Image")` creates an input component allowing users to upload a file.
*   `gr.Textbox(label="Uploaded File Path")` displays the path of the uploaded file.

To run this interface, uncomment `iface.launch()` in the code and execute the cell.

In [ ]:
import gradio as gr

ModuleNotFoundError: No module named 'gradio'

In [ ]:
import os
print(os.getcwd())
print(os.listdir("."))

c:\Users\Sahaana\OneDrive\Desktop\AI-PLANT-DISEASE-DETECTION
['.git', 'converted_savedmodel.zip', 'Plant_leave_diseases_dataset', 'Project 7-Plant Disease Detection-R.Sahaana.pdf', 'PROJECT_7_Notebook__AI_Powered_Plant_Disease_Detection (1).ipynb', 'README.md']


In [5]:
import zipfile, glob, os
zip_path = glob.glob("*.zip")[0]
zipfile.ZipFile(zip_path).extractall(".")
print(os.listdir("."))

['.git', 'converted_savedmodel.zip', 'labels.txt', 'model.savedmodel', 'Plant_leave_diseases_dataset', 'Project 7-Plant Disease Detection-R.Sahaana.pdf', 'PROJECT_7_Notebook__AI_Powered_Plant_Disease_Detection (1).ipynb', 'README.md']


In [6]:
%pip install tensorflow gradio google-genai pillow numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---

<center>
$END$
</center>

---